# Qualidade de `uso_mensal.csv`

Tabela agregada que o BI da Vitaliza usa nos painéis de engajamento: uma linha por usuário e mês, de jan/2025 a set/2026,
com treinos, minutos, dias ativos e aberturas do app.

| Critério | Pergunta |
|---|---|
| **Completude** | Tem linha para quem deveria ter? Há vazios, contas de teste, meses faltando no meio? |
| **Validade** | Ids, meses e números estão no formato e dentro de limites plausíveis? |
| **Consistência** | As colunas batem entre si? O agregado bate com o detalhe de `atividades.parquet`? |
| **Unicidade** | Há usuário-mês repetido? |
| **Temporalidade** | O volume por mês acompanha a base? Quem passa a ter linha, e quando? |

**Só diagnóstico:** nada em `dados/brutos/` muda e nenhum dado é corrigido. Cada achado sai com um tratamento sugerido,
que só vira regra no `decisoes.md` com aprovação.

**Arquivos cruzados:** `usuarios.csv` (cadastro e contas de teste); `atividades.parquet` (os mesmos treinos de 2026, por outro
sistema); o painel `assinaturas_mes` e os `trials` já limpos (quem era assinante ou estava em teste em cada mês).
**Número-prova:** a contagem de treinos de origem `app` do parquet, que bateu com este arquivo em 91,5% dos usuário-mês de 2026
no diagnóstico de `atividades.parquet`.

In [1]:
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 180)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
BRUTOS = ROOT / "dados" / "brutos"
PROCESSADOS = ROOT / "dados" / "processados"
SAIDAS = ROOT / "saidas"
INICIO_DADOS = pd.Period("2025-01", "M")
FIM_DADOS = pd.Period("2026-09", "M")
CORTE = pd.Timestamp("2026-09-30")
FIM_CORTE = CORTE + pd.Timedelta(days=1)

ARQUIVOS = ["uso_mensal.csv", "usuarios.csv", "atividades.parquet"]
LINHAS_DICIONARIO = 591_662
NUM = ["atividades", "atividades_ar_livre", "atividades_video", "minutos", "dias_ativos", "sessoes_app"]

def sha1(path):
    return hashlib.sha1(path.read_bytes()).hexdigest()

hash_inicio = {f: sha1(BRUTOS / f) for f in ARQUIVOS}

achados = []

def achado(criterio, checagem, resultado, linhas, tratamento, status=None):
    """status: ok (nada encontrado), esperado (encontrado, mas é comportamento do negócio) ou atenção."""
    if status is None:
        status = "ok" if linhas == 0 else "atenção"
    achados.append({"criterio": criterio, "checagem": checagem, "status": status,
                    "resultado": resultado, "linhas_afetadas": int(linhas), "tratamento_sugerido": tratamento})

## 0. Leitura

Tudo como texto; colunas derivadas com os números e o mês como período. As tabelas de assinatura já limpas dizem,
para cada usuário-mês, se a conta era assinante, estava em teste, já tinha sido assinante ou nunca assinou.

In [2]:
um = pd.read_csv(BRUTOS / "uso_mensal.csv", dtype=str, keep_default_na=False)
usuarios = pd.read_csv(BRUTOS / "usuarios.csv", dtype=str, keep_default_na=False,
                       usecols=["user_id", "email", "data_cadastro"])
assin = pd.read_parquet(PROCESSADOS / "assinaturas_limpa.parquet")
trials = pd.read_parquet(PROCESSADOS / "trials.parquet")
painel = pd.read_parquet(PROCESSADOS / "assinaturas_mes.parquet", columns=["assinatura_id", "user_id", "mes", "encerrou_no_mes", "iniciou_no_mes"])

assert len(um) == LINHAS_DICIONARIO
N = len(um)
print(f"uso_mensal.csv: {N:,} linhas x {um.shape[1]} colunas -> {list(um.columns)}")

X = um[NUM].apply(pd.to_numeric, errors="coerce")
um["p"] = pd.PeriodIndex(pd.to_datetime(um["mes"], format="%Y-%m", errors="coerce"), freq="M")
um["ano"] = um["mes"].str[:4]
u = usuarios.set_index("user_id")
cadastro = pd.to_datetime(u["data_cadastro"], format="%Y-%m-%d %H:%M:%S")

d0 = um["p"].dt.to_timestamp()
d1 = (um["p"] + 1).dt.to_timestamp()
ini = um["user_id"].map(assin.set_index("user_id")["inicio"])
fim = um["user_id"].map(assin.set_index("user_id")["data_fim"]).fillna(pd.Timestamp("2099-01-01"))
t_ini = um["user_id"].map(trials.set_index("user_id")["trial_inicio"])
t_fim = um["user_id"].map(trials.set_index("user_id")["data_desfecho"]).fillna(pd.Timestamp("2099-01-01"))
um["situacao"] = np.select(
    [ini.notna() & (ini < d1) & (fim >= d0), t_ini.notna() & (t_ini < d1) & (t_fim >= d0), ini.notna() & (fim < d0),
     ini.notna() & (ini >= d1)],
    ["assinante no mês", "em teste no mês", "ex-assinante", "assina depois"], "nunca assinou")
um.head()

uso_mensal.csv: 591,662 linhas x 8 colunas -> ['user_id', 'mes', 'atividades', 'atividades_ar_livre', 'atividades_video', 'minutos', 'dias_ativos', 'sessoes_app']


,user_id,mes,atividades,atividades_ar_livre,atividades_video,minutos,dias_ativos,sessoes_app,p,ano,situacao
0,u_f4d898b7,2025-01,1,0,1,28,1,2,2025-01,2025,assinante no mês
1,u_066eb85c,2025-01,11,1,10,457,10,28,2025-01,2025,assinante no mês
2,u_666cc36a,2025-01,9,1,8,377,8,24,2025-01,2025,assinante no mês
3,u_cdb02414,2025-01,8,1,7,273,7,20,2025-01,2025,assinante no mês
4,u_411ca38a,2025-01,4,3,1,173,4,9,2025-01,2025,assinante no mês


## 1. Completude

Espera-se nenhum vazio e nenhuma conta de teste. A pergunta central é quem deveria ter linha: todo assinante ativo no
mês (painel) deveria aparecer.

In [3]:
VAZIOS_DISFARCADOS = {"nan", "null", "none", "na", "n/a", "-"}
vazios = pd.DataFrame({
    "vazio": [(um[c].str.strip() == "").sum() for c in um.columns[:8]],
    "vazio_disfarcado": [um[c].str.strip().str.lower().isin(VAZIOS_DISFARCADOS).sum() for c in um.columns[:8]],
}, index=um.columns[:8])
display(vazios)
n_vazio = int(vazios.values.sum())
achado("Completude", "vazios por coluna (inclusive nan, null, -)", f"{n_vazio} células", n_vazio, "Nenhum")

teste = u["email"].str.strip().str.lower().str.endswith("@vitaliza.com.br")
duplicada = u["email"].str.contains(r"[A-Z]") & u["email"].str.strip().str.lower().duplicated(keep=False)
n_teste = int(um["user_id"].map(teste).sum())
n_dup = int(um["user_id"].map(duplicada).sum())
achado("Completude", "linhas de contas de teste (@vitaliza.com.br)", f"{n_teste}", n_teste, "Se houver: excluir")
achado("Completude", "linhas de contas duplicadas (e-mail com maiúscula, diagnóstico de usuarios.csv)", f"{n_dup}", n_dup,
       "Se houver: excluir junto com a conta")

,vazio,vazio_disfarcado
user_id,0,0
mes,0,0
atividades,0,0
atividades_ar_livre,0,0
atividades_video,0,0
minutos,0,0
dias_ativos,0,0
sessoes_app,0,0


### 1.1 Mês sem treino: em 2025 não tem linha

Hipótese: em 2025 o BI só gravava o usuário-mês **com pelo menos um treino**; em 2026 passou a gravar também mês sem treino,
e até mês sem nenhuma abertura do app. Se for isso, 2025 não tem nenhuma linha com `atividades = 0`.

In [4]:
tab = pd.DataFrame({
    "linhas": um.groupby("ano").size(),
    "atividades = 0": (X["atividades"] == 0).groupby(um["ano"]).sum(),
    "atividades = 0 e sessoes_app = 0": ((X["atividades"] == 0) & (X["sessoes_app"] == 0)).groupby(um["ano"]).sum(),
})
display(tab)
sem_treino_2025 = int(tab.loc["2025", "atividades = 0"])
zeradas_2026 = int(tab.loc["2026", "atividades = 0 e sessoes_app = 0"])
assert sem_treino_2025 == 0

achado("Completude", "mês sem treino em 2025",
       "0 linhas com atividades = 0 em 2025, contra 303,735 em 2026: em 2025 só existe linha no mês com treino",
       0, "Em 2025, tratar usuário-mês sem linha como 0 treino, 0 minuto e 0 dia ativo; sessoes_app desse mês é desconhecida "
       "(não usar sessoes_app de 2025 para medir quem parou de abrir o app)", status="atenção")
achado("Completude", "linhas com tudo zerado (sem treino e sem abrir o app)",
       f"{zeradas_2026:,} linhas, todas de 2026", zeradas_2026,
       "Não é erro: em 2026 o arquivo grava também o mês sem uso. Manter", status="esperado")

,linhas,atividades = 0,atividades = 0 e sessoes_app = 0
ano,,,
2025,152230,0,0
2026,439432,303735,41493


### 1.2 Assinante ativo sem linha

Cada linha do painel (assinatura × mês) deveria ter uma linha aqui. Em 2025, o que falta deveria ser mês sem treino
(seção 1.1). Em 2026, mês sem treino tem linha: o que falta precisa de outra explicação. O parquet diz se houve treino.

In [5]:
at = pd.read_parquet(BRUTOS / "atividades.parquet", columns=["atividade_id", "user_id", "inicio", "duracao_min", "origem"])
at["t"] = pd.to_datetime(at["inicio"], format="%Y-%m-%d %H:%M:%S")
# origem app e Apple Health em horário de Brasília; Garmin em UTC sem o Z (decisão de 31/10/2026): -3h
at.loc[at["origem"] == "garmin", "t"] -= pd.Timedelta(hours=3)
at["mes"] = at["t"].dt.strftime("%Y-%m")
treinos_mes = at.groupby(["user_id", "mes"]).size()
treinos_app_mes = at.loc[at["origem"] == "app"].groupby(["user_id", "mes"]).size()

pn = painel.assign(mes=painel["mes"].dt.strftime("%Y-%m"))
pn = pn.loc[pn["mes"] >= "2025-01"]
chave_um = pd.MultiIndex.from_frame(um[["user_id", "mes"]])
chave_pn = pd.MultiIndex.from_frame(pn[["user_id", "mes"]])
pn["tem_linha"] = chave_pn.isin(chave_um)
pn["treinos_parquet"] = treinos_mes.reindex(chave_pn).fillna(0).values
pn["ano"] = pn["mes"].str[:4]

falta = pn.loc[~pn["tem_linha"]]
display(pd.DataFrame({
    "linhas do painel": pn.groupby("ano").size(),
    "sem linha no uso": falta.groupby("ano").size(),
    "  das quais terminam no mês": falta.groupby("ano")["encerrou_no_mes"].sum(),
    "  das quais começam no mês": falta.groupby("ano")["iniciou_no_mes"].sum(),
    "  com treino no parquet (2026)": falta.loc[falta["ano"] == "2026"].groupby("ano")["treinos_parquet"].apply(lambda s: (s > 0).sum()),
}).fillna("").T)

f25, f26 = (falta["ano"] == "2025").sum(), (falta["ano"] == "2026").sum()
f26_com_treino = int(((falta["ano"] == "2026") & (falta["treinos_parquet"] > 0)).sum())
achado("Completude", "assinante ativo no mês sem linha, 2025", f"{f25:,} de {(pn['ano'] == '2025').sum():,} linhas do painel; "
       "coerente com 2025 só gravar mês com treino", f25,
       "Tratar como mês sem treino (0), com sessoes_app desconhecida", status="atenção")
achado("Completude", "assinante ativo no mês sem linha, 2026",
       f"{f26:,} de {(pn['ano'] == '2026').sum():,} linhas do painel; {f26_com_treino:,} delas têm treino no parquet", f26,
       "Em 2026, medir uso pelo parquet (decisão de 31/10/2026); para sessoes_app, mês sem linha fica como desconhecido. "
       "Perguntar ao dono do dado por que a linha falta", status="atenção")

pq_sem_linha = treinos_mes.index.difference(chave_um)
achado("Completude", "usuário-mês com treino no parquet e sem linha no uso (2026)",
       f"{len(pq_sem_linha):,} usuário-mês de {pq_sem_linha.get_level_values(0).nunique():,} contas", len(pq_sem_linha),
       "Em 2026, partir do parquet para contar treinos", status="atenção")

ano,2025,2026
linhas do painel,146714,135859
sem linha no uso,23249,3690
das quais terminam no mês,4453,764
das quais começam no mês,485,54
com treino no parquet (2026),,933.0


### 1.3 Meses faltando no meio da sequência

Num agregado mensal, a conta que aparece em março e maio e some em abril tem um buraco. Em 2025 o buraco é mês sem treino;
em 2026 ele acontece mesmo com mês zerado gravado.

In [6]:
s = um.sort_values(["user_id", "p"])
salto = s.groupby("user_id")["p"].diff().apply(lambda d: d.n if pd.notna(d) else 1)
buracos = s.loc[salto > 1].assign(ano=lambda x: x["ano"])
print(f"buracos: {len(buracos):,} em {buracos['user_id'].nunique():,} contas; por ano do mês que volta: "
      f"{buracos['ano'].value_counts().sort_index().to_dict()}")
achado("Completude", "meses faltando no meio da sequência de cada conta",
       f"{len(buracos):,} buracos em {buracos['user_id'].nunique():,} contas", len(buracos),
       "Ao montar série por conta, preencher o mês faltante: 0 treino; sessoes_app desconhecida", status="atenção")

buracos: 59,438 em 34,546 contas; por ano do mês que volta: {'2025': 5962, '2026': 53476}


## 2. Validade

Espera-se `user_id` no padrão e cadastrado, `mes` de 2025-01 a 2026-09 e números inteiros não negativos, com limites plausíveis.

In [7]:
fora_padrao = (~um["user_id"].str.fullmatch(r"u_[0-9a-f]{8}")).sum()
sem_cadastro = (~um["user_id"].isin(u.index)).sum()
achado("Validade", "user_id fora do padrão ou sem cadastro", f"{fora_padrao} fora do padrão; {sem_cadastro} sem cadastro",
       fora_padrao + sem_cadastro, "Se houver: excluir e registrar")

mes_invalido = um["p"].isna().sum()
fora_periodo = ((um["p"] < INICIO_DADOS) | (um["p"] > FIM_DADOS)).sum()
achado("Validade", "mes fora do formato AAAA-MM ou fora de jan/2025 a set/2026",
       f"{mes_invalido} fora do formato; {fora_periodo} fora do período", mes_invalido + fora_periodo, "Se houver: excluir")

nao_inteiro = int((X.isna() | (X % 1 != 0)).sum().sum())
negativo = int((X < 0).sum().sum())
achado("Validade", "número que não converte, não é inteiro ou é negativo", f"{nao_inteiro} não inteiros; {negativo} negativos",
       nao_inteiro + negativo, "Nenhum")

dias_no_mes = um["p"].dt.days_in_month
dias_demais = (X["dias_ativos"] > dias_no_mes).sum()
achado("Validade", "dias_ativos maior que os dias do mês", f"{dias_demais} linhas", dias_demais, "Nenhum")

min_por_treino = (X["minutos"] / X["atividades"]).where(X["atividades"] > 0)
print(f"minutos por treino: {min_por_treino.describe(percentiles=[.01, .5, .99]).round(1).to_dict()}")
fora_min = ((min_por_treino < 5) | (min_por_treino > 300)).sum()
achado("Validade", "minutos por treino fora de 5 a 300", f"{fora_min} linhas (observado: {min_por_treino.min():.0f} a {min_por_treino.max():.0f})",
       fora_min, "Nenhum")

print(f"sessoes_app: {X['sessoes_app'].describe(percentiles=[.5, .99]).round(1).to_dict()}")
sessoes_demais = (X["sessoes_app"] > 10 * dias_no_mes).sum()
achado("Validade", "mais de 10 aberturas do app por dia, na média do mês", f"{sessoes_demais} linhas (máximo {X['sessoes_app'].max():.0f} no mês)",
       sessoes_demais, "Nenhum")

minutos por treino: {'count': 287927.0, 'mean': 44.3, 'std': 16.2, 'min': 12.0, '1%': 20.4, '50%': 41.0, '99%': 99.3, 'max': 222.0}
sessoes_app: {'count': 591662.0, 'mean': 7.2, 'std': 10.7, 'min': 0.0, '50%': 2.0, '99%': 50.0, 'max': 153.0}


## 3. Consistência

Regras entre colunas da mesma linha; depois, o agregado contra o detalhe do parquet em 2026.

In [8]:
regras = {
    "ar livre + vídeo diferente do total de treinos": (X["atividades_ar_livre"] + X["atividades_video"] != X["atividades"]),
    "dias ativos maior que o número de treinos": (X["dias_ativos"] > X["atividades"]),
    "treino sem dia ativo, ou dia ativo sem treino": ((X["atividades"] > 0) != (X["dias_ativos"] > 0)),
    "treino sem minutos, ou minutos sem treino": ((X["atividades"] > 0) != (X["minutos"] > 0)),
    "treino registrado sem nenhuma abertura do app": ((X["atividades"] > 0) & (X["sessoes_app"] == 0)),
    "mês anterior ao mês de cadastro": (um["p"] < um["user_id"].map(cadastro).dt.to_period("M")),
}
for nome, m in regras.items():
    achado("Consistência", nome, f"{int(m.sum())} linhas", int(m.sum()), "Nenhum")
pd.Series({k: int(v.sum()) for k, v in regras.items()}, name="linhas").to_frame()

,linhas
ar livre + vídeo diferente do total de treinos,0
dias ativos maior que o número de treinos,0
"treino sem dia ativo, ou dia ativo sem treino",0
"treino sem minutos, ou minutos sem treino",0
treino registrado sem nenhuma abertura do app,0
mês anterior ao mês de cadastro,0


### 3.1 Agregado × parquet (2026)

O diagnóstico de `atividades.parquet` mostrou que este arquivo conta só os treinos registrados no app. Aqui a comparação
é refeita por usuário-mês, com contagem, minutos e dias ativos, e duas hipóteses para a diferença que sobra:
mês fechado em UTC (treinos da noite do último dia) e treinos com duração 0 fora da conta.

In [9]:
app = at.loc[at["origem"] == "app"].copy()
g = app.groupby(["user_id", "mes"]).agg(n_pq=("atividade_id", "size"), min_pq=("duracao_min", "sum"),
                                        dias_pq=("t", lambda s: s.dt.normalize().nunique()))
app["mes_utc"] = (app["t"] + pd.Timedelta(hours=3)).dt.strftime("%Y-%m")
g["n_pq_utc"] = app.groupby(["user_id", "mes_utc"]).size().rename_axis(["user_id", "mes"])
g["n_pq_sem_dur0"] = app.loc[app["duracao_min"] > 0].groupby(["user_id", "mes"]).size()

c = (um.loc[um["ano"] == "2026", ["user_id", "mes"]].join(X).set_index(["user_id", "mes"])
     .join(g, how="outer").fillna(0))
c["total_pq"] = treinos_mes.reindex(c.index).fillna(0).values

comp = pd.DataFrame({
    "usuário-mês em que bate": {
        "uso × todos os treinos do parquet": (c["atividades"] == c["total_pq"]).mean(),
        "uso × treinos do app (número-prova)": (c["atividades"] == c["n_pq"]).mean(),
        "uso × treinos do app, mês em UTC": (c["atividades"] == c["n_pq_utc"]).mean(),
        "uso × treinos do app sem duração 0": (c["atividades"] == c["n_pq_sem_dur0"]).mean(),
    }}).round(4)
display(comp)
dif = (c["atividades"] - c["n_pq"]).astype(int)
display(dif.value_counts().sort_index().rename("usuário-mês").to_frame().T)

bate = c.loc[c["atividades"] == c["n_pq"]]
dias_iguais = (bate["dias_ativos"] == bate["dias_pq"]).mean()
min_iguais = (bate["minutos"].round() == bate["min_pq"].round()).mean()
razao_min = (bate["minutos"] / bate["min_pq"]).where(bate["min_pq"] > 0)
print(f"nos usuário-mês em que a contagem bate: dias ativos iguais em {dias_iguais:.1%}; minutos iguais em {min_iguais:.1%} "
      f"(razão uso ÷ parquet: p1 {razao_min.quantile(.01):.2f}, mediana {razao_min.median():.2f}, p99 {razao_min.quantile(.99):.2f})")
assert abs(comp.loc["uso × treinos do app (número-prova)"].iloc[0] - 0.915) < 0.001

n_div = int((dif != 0).sum())
achado("Consistência", "atividades × treinos de origem app no parquet (2026)",
       f"bate em {comp.iloc[1, 0]:.1%} dos usuário-mês; {n_div:,} divergem de ±1 a ±8, de forma simétrica, em todos os meses. "
       f"Não é o fuso do mês ({comp.iloc[2, 0]:.1%}) nem a duração 0 ({comp.iloc[3, 0]:.1%})", n_div,
       "Em 2026, contar treinos pelo parquet. A diferença fica em aberto com o dono do dado")
achado("Consistência", "atividades × todos os treinos do parquet (2026)",
       f"bate em {comp.iloc[0, 0]:.1%}: os treinos sincronizados do relógio ficam de fora (decisão de 31/10/2026)",
       int((c["atividades"] != c["total_pq"]).sum()), "Já decidido: em 2026, medir uso pelo parquet", status="atenção")
n_min = int((bate["minutos"].round() != bate["min_pq"].round()).sum())
achado("Consistência", "minutos × soma da duração no parquet, onde a contagem bate (2026)",
       f"iguais em {min_iguais:.1%}; razão mediana {razao_min.median():.2f} (p1 {razao_min.quantile(.01):.2f}, p99 {razao_min.quantile(.99):.2f})",
       n_min, "Em 2026, somar minutos pelo parquet; em 2025, usar minutos daqui sabendo que é aproximado")

,usuário-mês em que bate
uso × todos os treinos do parquet,0.8649
uso × treinos do app (número-prova),0.9149
"uso × treinos do app, mês em UTC",0.9126
uso × treinos do app sem duração 0,0.9105


,-8,-7,-6,-5,-4,-3,-2,-1,0,1,2,3,4,5,6,7
usuário-mês,8,22,59,188,561,1615,4401,15862,402662,11033,2919,613,123,29,6,1


nos usuário-mês em que a contagem bate: dias ativos iguais em 99.8%; minutos iguais em 74.6% (razão uso ÷ parquet: p1 0.40, mediana 0.94, p99 2.49)


## 4. Unicidade

In [10]:
rep_chave = um.duplicated(["user_id", "mes"]).sum()
rep_linha = um.duplicated(subset=list(um.columns[:8])).sum()
achado("Unicidade", "user_id + mes repetido", f"{rep_chave}", rep_chave, "Se houver: investigar antes de somar")
achado("Unicidade", "linha inteira repetida", f"{rep_linha}", rep_linha, "Se houver: manter a primeira")

## 5. Temporalidade

O volume salta em jan/2026. Antes de chamar de falha, a tabela abaixo separa quem tem linha pela situação na Vitaliza no mês.

In [11]:
vol = um.groupby("mes").size().rename("linhas").to_frame()
vol["variação %"] = (100 * vol["linhas"].pct_change()).round(1)
vol["cadastros acumulados"] = [((cadastro < (pd.Period(m, "M") + 1).to_timestamp()) & ~teste).sum() for m in vol.index]
vol["assinantes no painel"] = pn.groupby("mes").size().reindex(vol.index)
display(vol.T)

sit = pd.crosstab(um["mes"], um["situacao"])
display(sit.T)

# só linhas com treino, para comparar os dois anos no mesmo critério de 2025
com_treino = um.loc[X["atividades"] > 0]
por_mes = pd.crosstab(com_treino["situacao"], com_treino["ano"])
por_mes = pd.DataFrame({"2025 (média por mês)": (por_mes["2025"] / 12).round(0),
                        "2026 (média por mês)": (por_mes["2026"] / 9).round(0)}).rename_axis("linhas com treino, por situação")
display(por_mes)

saltos = vol.loc[vol["variação %"] > 50]
achado("Temporalidade", "mês com salto de volume acima de 50%",
       f"{', '.join(f'{m}: +{v:.0f}%' for m, v in saltos['variação %'].items())}. Só as linhas com treino dão "
       f"{com_treino.loc[com_treino['ano'] == '2025'].shape[0] / 12:,.0f} por mês em 2025 e "
       f"{com_treino.loc[com_treino['ano'] == '2026'].shape[0] / 9:,.0f} em 2026, com a mesma divisão por situação: o salto é "
       f"2026 passar a gravar o mês sem treino (ex-assinantes e contas grátis paradas)",
       int(saltos["linhas"].sum()),
       "Para comparar 2025 com 2026, usar só linhas com treino (ou preencher 2025 com zero); não comparar contagem de linhas, "
       "média por conta nem sessoes_app entre os anos", status="atenção")

assina_depois = int((um["situacao"] == "assina depois").sum())
achado("Temporalidade", "conta com linha antes de começar o teste ou a assinatura",
       f"{assina_depois} linhas: em 2025 não aparece uso de conta grátis que só assinaria depois", assina_depois,
       "Nenhum", status="ok" if assina_depois == 0 else "atenção")
achado("Temporalidade", "colunas sem data dentro do mês",
       "o mês esconde quando os treinos aconteceram; uma variável do mês do corte (set/2026) inclui o mês inteiro",
       0, "Ao usar como variável de um mês, pegar só meses fechados antes da data da previsão", status="esperado")

mes,2025-01,2025-02,2025-03,2025-04,2025-05,2025-06,2025-07,2025-08,2025-09,2025-10,2025-11,2025-12,2026-01,2026-02,2026-03,2026-04,2026-05,2026-06,2026-07,2026-08,2026-09
linhas,11441.0,11257.0,12005.0,12218.0,12419.0,12264.0,12460.0,13003.0,13452.0,14048.0,14039.0,13624.0,41130.0,43279.0,45309.0,47127.0,48909.0,50257.0,52234.0,54622.0,56565.0
variação %,NaN,-1.6,6.6,1.8,1.6,-1.2,1.6,4.4,3.5,4.4,-0.1,-3.0,201.9,5.2,4.7,4.0,3.8,2.8,3.9,4.6,3.6
cadastros acumulados,19798.0,22021.0,24336.0,26244.0,28127.0,29993.0,31968.0,34167.0,36483.0,38836.0,40899.0,42664.0,46176.0,48732.0,51166.0,53359.0,55478.0,57442.0,59618.0,62190.0,64769.0
assinantes no painel,9963.0,10552.0,11136.0,11463.0,11868.0,12205.0,12497.0,12800.0,13123.0,13519.0,13749.0,13839.0,14701.0,14947.0,15242.0,15277.0,15140.0,14934.0,14989.0,15159.0,15470.0


mes,2025-01,2025-02,2025-03,2025-04,2025-05,2025-06,2025-07,2025-08,2025-09,2025-10,2025-11,2025-12,2026-01,2026-02,2026-03,2026-04,2026-05,2026-06,2026-07,2026-08,2026-09
situacao,,,,,,,,,,,,,,,,,,,,,
assinante no mês,8765,8866,9486,9741,10053,10174,10319,10648,10963,11424,11590,11436,14355,14541,14872,14839,14761,14468,14523,14752,15058
em teste no mês,1045,799,757,681,632,585,683,743,810,839,698,596,1491,1329,1233,1087,1009,984,1094,1250,1256
ex-assinante,61,74,76,83,89,77,85,115,98,105,114,83,8058,8843,9542,10430,11402,12285,13107,13900,14482
nunca assinou,1570,1518,1686,1713,1645,1428,1373,1497,1581,1680,1637,1509,17226,18566,19662,20771,21737,22520,23510,24720,25769


,2025 (média por mês),2026 (média por mês)
"linhas com treino, por situação",,
assinante no mês,10289.0,12323.0
em teste no mês,739.0,856.0
ex-assinante,88.0,129.0
nunca assinou,1570.0,1769.0


## 6. Resumo

In [12]:
resumo = pd.DataFrame(achados)
resumo["% do arquivo"] = (100 * resumo["linhas_afetadas"] / N).round(2)
resumo.to_csv(SAIDAS / "qualidade_uso_mensal.csv", index=False)
print(resumo["status"].value_counts().to_dict())
resumo

{'ok': 18, 'atenção': 9, 'esperado': 2}


,criterio,checagem,status,resultado,linhas_afetadas,tratamento_sugerido,% do arquivo
0,Completude,"vazios por coluna (inclusive nan, null, -)",ok,0 células,0,Nenhum,0.00
1,Completude,linhas de contas de teste (@vitaliza.com.br),ok,0,0,Se houver: excluir,0.00
2,Completude,linhas de contas duplicadas (e-mail com maiúsc...,ok,0,0,Se houver: excluir junto com a conta,0.00
3,Completude,mês sem treino em 2025,atenção,"0 linhas com atividades = 0 em 2025, contra 30...",0,"Em 2025, tratar usuário-mês sem linha como 0 t...",0.00
4,Completude,linhas com tudo zerado (sem treino e sem abrir...,esperado,"41,493 linhas, todas de 2026",41493,Não é erro: em 2026 o arquivo grava também o m...,7.01
5,Completude,"assinante ativo no mês sem linha, 2025",atenção,"23,249 de 146,714 linhas do painel; coerente c...",23249,"Tratar como mês sem treino (0), com sessoes_ap...",3.93
6,Completude,"assinante ativo no mês sem linha, 2026",atenção,"3,690 de 135,859 linhas do painel; 933 delas t...",3690,"Em 2026, medir uso pelo parquet (decisão de 31...",0.62
7,Completude,usuário-mês com treino no parquet e sem linha ...,atenção,"2,173 usuário-mês de 1,998 contas",2173,"Em 2026, partir do parquet para contar treinos",0.37
8,Completude,meses faltando no meio da sequência de cada conta,atenção,"59,438 buracos em 34,546 contas",59438,"Ao montar série por conta, preencher o mês fal...",10.05
9,Validade,user_id fora do padrão ou sem cadastro,ok,0 fora do padrão; 0 sem cadastro,0,Se houver: excluir e registrar,0.00


In [13]:
hash_fim = {f: sha1(BRUTOS / f) for f in ARQUIVOS}
assert hash_fim == hash_inicio
print("dados/brutos/ inalterado:", hash_fim == hash_inicio)

dados/brutos/ inalterado: True
